In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

bronze_schema = StructType([
    StructField("indicator", StructType([
        StructField("id", StringType(), True),
        StructField("value", StringType(), True)
    ]), True),
    StructField("country", StructType([
        StructField("id", StringType(), True),
        StructField("value", StringType(), True)
    ]), True),
    StructField("countryiso3code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("obs_status", StringType(), True),
    StructField("decimal", StringType(), True)
])

In [0]:
bronze_df = (
    spark.read
    .schema(bronze_schema)
    .option("multiline", "true")
    .json("/Workspace/Users/l242585@lhr.nu.edu.pk/World_Bank_Analytics/full_load_part1.json")
)

# Agar teeno parts combine karni hain
bronze_df2 = spark.read.schema(bronze_schema).option("multiline", "true").json("/Workspace/Users/l242585@lhr.nu.edu.pk/World_Bank_Analytics/full_load_part1.json/")
bronze_df3 = spark.read.schema(bronze_schema).option("multiline", "true").json("/Workspace/Users/l242585@lhr.nu.edu.pk/World_Bank_Analytics/full_load_part1.json")

bronze_df = bronze_df.unionByName(bronze_df2).unionByName(bronze_df3)

print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = bronze_df.withColumn("load_timestamp", current_timestamp())
bronze_df.display()

In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_indicator_values")

In [0]:
from pyspark.sql.functions import col, when

silver_df = (
    bronze_df
    .select(
        col("country.id").alias("country_code"),
        col("country.value").alias("country_name"),
        col("indicator.id").alias("indicator_code"),
        col("indicator.value").alias("indicator_name"),
        col("date").cast(IntegerType()).alias("year"),
        col("value").cast(DoubleType()).alias("value"),
        col("load_timestamp")
    )
    .withColumn("is_reported", when(col("value").isNotNull(), True).otherwise(False))
    .dropDuplicates(["country_code", "indicator_code", "year"])
)

silver_df.printSchema()
print(f"Silver row count: {silver_df.count()}")

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_indicator_values")